# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema JSON-LD)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata with mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by `@id` fields.

In [ ]:
# List all record sets and their field @ids

record_set_objs = getattr(metadata, 'recordSet', [])

if not record_set_objs:
    print("No record sets found in metadata. Attempting to extract record sets from attached distributions...")
    # Sometimes datasets do not list 'recordSet' at top-level, but under distribution or elsewhere.
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            attrs = dist.__dict__ if hasattr(dist, '__dict__') else {}
            for k, v in attrs.items():
                if k == 'recordSet':
                    record_set_objs = v
    
if not record_set_objs:
    # Try ._json() - fallback for advanced users
    raw_meta = dataset.metadata.to_json() if hasattr(dataset.metadata, 'to_json') else {}
    record_set_objs = raw_meta.get('recordSet', [])
    if not record_set_objs and 'distribution' in raw_meta:
        dists = raw_meta['distribution']
        for dist in dists:
            if 'recordSet' in dist:
                record_set_objs += dist['recordSet']

# Print overview for each record set
record_set_ids = []
for recset in record_set_objs:
    recset_id = getattr(recset, '@id', None) or recset.get('@id') if isinstance(recset, dict) else recset
    record_set_ids.append(recset_id)
    print(f"RecordSet @id: {recset_id}")
    # List fields/columns (usually in 'field' or 'column')
    fields = getattr(recset, 'field', None) or getattr(recset, 'column', None) or recset.get('field') if isinstance(recset, dict) else []
    if fields:
        print("  Fields/columns @id:")
        for f in fields:
            field_id = getattr(f, '@id', None) or f.get('@id') if isinstance(f, dict) else f
            print(f"    {field_id}")
    else:
        print("  (No fields/columns listed)")
if not record_set_ids:
    print("No explicit record sets found. Please check the Croissant file for more details about data access.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All fields are referenced by their `@id`s as shown in the overview above.

In [ ]:
# Prepare to extract data from each record set

# If no record sets were detected, guess a main record set id
if not record_set_ids:
    # Example fallback guessed from likely usage
    # Please edit the below to reflect a valid `@id` if known
    record_set_ids = [None]  # User must fill a valid `@id` from schema docs

dataframes = {}

for record_set_id in record_set_ids:
    if not record_set_id:
        continue
    print(f"Loading record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# If only one DataFrame loaded, pick it for further analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Main record set for analysis: {main_record_set_id}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Reference numeric and group fields using their `@id`. You may need to adjust these based on your schema.

In [ ]:
# The following assumes at least one numeric field exists.
# Please update `numeric_field_id` and `group_field_id` according to your record set schema.

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Available columns in main DataFrame: {df.columns.tolist()}")

    # Example: Try to select likely numeric and group fields
    # You may want to edit these to match the `@id` from your overview
    numeric_field_id = None
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
    
    if numeric_field_id is None:
        print("No numeric field found, please specify the numeric_field_id from schema.")
    else:
        threshold = df[numeric_field_id].mean()  # Set a threshold as an example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Example group field
        group_field_id = None
        # Try to automatically find a likely categorical/group column
        non_numeric = [col for col in df.columns if df[col].dtype == object]
        if non_numeric:
            group_field_id = non_numeric[0]

        if group_field_id in filtered_df.columns:
            try:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
                print(f"Grouped data by {group_field_id}, showing mean {numeric_field_id}:")
                display(grouped_df.head())
            except Exception as e:
                print(f"Could not group by {group_field_id}: {e}")
        else:
            print(f"No valid group field found; skipping group aggregation.")
else:
    print("No main record set DataFrame found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: Display histogram of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(data=df, x=numeric_field_id, kde=True, bins=30)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Optional: boxplot for visualization by a categorical field
    if group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**In this notebook, we loaded the FAIR^2 Croissant metadata and records for the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset, providing a step-by-step exploration using the `mlcroissant` library.**

- We loaded the metadata and found available record sets and fields (all referenced by their `@id`).
- We extracted data into pandas DataFrames using `mlcroissant` record iteration.
- We performed exploratory analysis, including numeric filtering, normalization, and simple group aggregations, with all operations referencing fields by `@id`.
- Basic visualizations demonstrated the underlying data distributions and categorical splits where possible.

For deeper analysis, examine the Croissant schema and documentation to refine `record_set` and field `@id` usage to target biological or mass spectrometry phenomena of interest.